# Data Partition cho Federated Learning

Notebook tạo file partition `.pt` phân chia dữ liệu MNIST cho các client, hỗ trợ 2 chế độ:

| `PARTITION_TYPE` | Mô tả |
|---|---|
| `dirichlet` | Non-IID theo phân phối Dirichlet(α). α nhỏ → skew cao, α lớn → gần IID. |
| `noniid_2`, `noniid_3`, … | Mỗi client nhận đúng **N class** (phân đều trong nội bộ class). Số sau `_` là số class mỗi client. |

## ⚙️ Cấu hình

In [ ]:
# ── Tham số chính ──────────────────────────────────────────────────────────────
PARTITION_TYPE = "dirichlet"   # "dirichlet" | "noniid_2" | "noniid_3" | ...
NUM_CLIENTS    = 25
SEED           = 42
MIN_SAMPLES    = 10            # Số mẫu tối thiểu mỗi client (chỉ áp dụng Dirichlet)

# Tham số cho Dirichlet
ALPHA = 0.5                    # Concentration parameter

# ── Đường dẫn ─────────────────────────────────────────────────────────────────
PROJECT_ROOT = None            # None = thư mục hiện tại; hoặc đặt thành "/path/to/project"
OVERWRITE    = False           # Ghi đè file nếu đã tồn tại
NO_PLOT      = False           # True = bỏ qua bước vẽ biểu đồ

## 📦 Imports & Helpers

In [ ]:
import csv
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets


def _resolve_project_root(root_override) -> Path:
    """Trả về PROJECT_ROOT: ưu tiên tham số, sau đó Colab Drive, cuối cùng là CWD."""
    if root_override is not None:
        return Path(root_override)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return Path("/content/drive/MyDrive/FLasIRL(new)")
    except ImportError:
        return Path.cwd()


def _parse_partition_type(partition_type: str) -> tuple[str, int | None]:
    """
    Parse PARTITION_TYPE:
      - "dirichlet"  → ("dirichlet", None)
      - "noniid_2"   → ("noniid", 2)
      - "noniid_10"  → ("noniid", 10)
    """
    if partition_type == "dirichlet":
        return "dirichlet", None
    m = re.fullmatch(r"noniid_(\d+)", partition_type)
    if m:
        return "noniid", int(m.group(1))
    raise ValueError(
        f"PARTITION_TYPE không hợp lệ: '{partition_type}'. "
        "Dùng 'dirichlet' hoặc 'noniid_N' (ví dụ: 'noniid_2', 'noniid_3')."
    )

## 🔧 Hàm tạo partition

In [ ]:
def generate_dirichlet_partition(
    train_labels: torch.Tensor,
    num_clients: int = 25,
    alpha: float = 0.5,
    seed: int = 42,
    min_samples: int = 10,
    max_retries: int = 100,
) -> tuple[list[torch.Tensor], torch.Tensor]:
    """
    Phân chia chỉ số theo phân phối Dirichlet-Multinomial trên từng class.

    Returns:
        client_indices      : list[Tensor] – chỉ số mẫu của mỗi client.
        client_class_counts : Tensor (num_clients, num_classes) – số mẫu theo class.
    """
    labels_np   = train_labels.cpu().numpy()
    num_samples = len(labels_np)
    classes     = np.unique(labels_np)
    num_classes = len(classes)

    for attempt in range(max_retries):
        rng     = np.random.default_rng(seed + attempt)
        buckets = [[] for _ in range(num_clients)]

        for c in classes:
            idx_c       = np.where(labels_np == c)[0]
            rng.shuffle(idx_c)
            proportions = rng.dirichlet(np.full(num_clients, alpha))
            counts      = rng.multinomial(len(idx_c), proportions)
            start = 0
            for cid, cnt in enumerate(counts):
                if cnt > 0:
                    buckets[cid].extend(idx_c[start : start + cnt])
                    start += cnt

        if min(len(b) for b in buckets) >= min_samples:
            break
    else:
        raise RuntimeError(
            f"Không thể tạo partition Dirichlet (alpha={alpha}, min_samples={min_samples}) "
            f"sau {max_retries} lần thử. Hãy giảm min_samples hoặc tăng alpha."
        )

    client_indices      = []
    client_class_counts = torch.zeros((num_clients, num_classes), dtype=torch.long)
    for cid, indices in enumerate(buckets):
        arr = np.array(indices, dtype=np.int64)
        rng.shuffle(arr)
        t   = torch.tensor(arr, dtype=torch.long)
        client_indices.append(t)
        client_class_counts[cid] = torch.bincount(train_labels[t], minlength=num_classes)

    assert sum(len(x) for x in client_indices) == num_samples, "Tổng mẫu không khớp!"
    assert len(torch.cat(client_indices).unique()) == num_samples, "Có mẫu bị trùng lặp!"
    return client_indices, client_class_counts

In [ ]:
def generate_noniid_partition(
    train_labels: torch.Tensor,
    num_clients: int = 25,
    classes_per_client: int = 2,
    seed: int = 42,
) -> tuple[list[torch.Tensor], torch.Tensor]:
    """
    Mỗi client nhận đúng `classes_per_client` class, phân đều trong nội bộ từng class.

    Returns:
        client_indices      : list[Tensor] – chỉ số mẫu của mỗi client.
        client_class_counts : Tensor (num_clients, num_classes) – số mẫu theo class.
    """
    labels_np   = train_labels.cpu().numpy()
    classes     = np.unique(labels_np)
    num_classes = len(classes)
    rng         = np.random.default_rng(seed)

    if not (1 <= classes_per_client <= num_classes):
        raise ValueError(
            f"classes_per_client phải nằm trong [1, {num_classes}], nhận được {classes_per_client}."
        )

    # Gán class theo vòng tròn rồi xáo trộn thứ tự client
    client_classes = np.array(
        [classes[(classes_per_client * i + j) % num_classes]
         for i in range(num_clients)
         for j in range(classes_per_client)]
    ).reshape(num_clients, classes_per_client)
    rng.shuffle(client_classes, axis=0)

    buckets = [[] for _ in range(num_clients)]
    for class_id in classes:
        class_clients = np.where(client_classes == class_id)[0]
        class_indices = np.flatnonzero(labels_np == class_id)
        rng.shuffle(class_indices)
        for cid, chunk in zip(class_clients, np.array_split(class_indices, len(class_clients))):
            buckets[cid].extend(chunk.tolist())

    client_indices      = []
    client_class_counts = torch.zeros((num_clients, num_classes), dtype=torch.long)
    for cid, indices in enumerate(buckets):
        arr = np.asarray(indices, dtype=np.int64)
        rng.shuffle(arr)
        t   = torch.tensor(arr, dtype=torch.long)
        client_indices.append(t)
        client_class_counts[cid] = torch.bincount(train_labels[t], minlength=num_classes)

    assert sum(len(x) for x in client_indices) == len(labels_np), "Tổng mẫu không khớp!"
    assert len(torch.cat(client_indices).unique()) == len(labels_np), "Có mẫu bị trùng lặp!"
    assert torch.all((client_class_counts > 0).sum(dim=1) == classes_per_client), \
        "Có client không nhận đủ số class!"
    return client_indices, client_class_counts

## 📊 Hàm vẽ biểu đồ

In [ ]:
def plot_distribution(
    client_class_counts: torch.Tensor,
    title: str,
    chart_path: Path,
) -> None:
    """Vẽ stacked-bar chart phân bố class của các client và lưu ảnh."""
    num_clients  = client_class_counts.shape[0]
    num_classes  = client_class_counts.shape[1]
    client_sizes = client_class_counts.sum(dim=1).cpu().numpy()
    colors = plt.cm.tab10(np.linspace(0, 1, num_classes))

    fig, ax = plt.subplots(figsize=(max(10, num_clients // 2), 5))
    bottom  = np.zeros(num_clients)

    for cls in range(num_classes):
        vals = client_class_counts[:, cls].cpu().numpy()
        ax.bar(np.arange(num_clients), vals, bottom=bottom,
               label=f"Class {cls}", color=colors[cls],
               edgecolor="black", linewidth=0.4)
        bottom += vals

    ax.set_title(
        f"{title}\nMin={client_sizes.min()}, Max={client_sizes.max()}, Mean={client_sizes.mean():.1f}",
        fontsize=12,
    )
    ax.set_xlabel("Client ID", fontsize=11)
    ax.set_ylabel("Số mẫu", fontsize=11)
    ax.set_xticks(np.arange(num_clients))
    ax.set_xticklabels(range(num_clients), fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.legend(ncol=num_classes, loc="upper center",
              bbox_to_anchor=(0.5, -0.18), frameon=True, fontsize=9)
    fig.tight_layout()

    chart_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Biểu đồ đã lưu: {chart_path}")

## 🚀 Chạy partition

In [ ]:
# ── Parse & validate PARTITION_TYPE ───────────────────────────────────────────
ptype, classes_per_client = _parse_partition_type(PARTITION_TYPE)

# ── Đường dẫn ─────────────────────────────────────────────────────────────────
project_root  = _resolve_project_root(PROJECT_ROOT)
data_dir      = project_root / "data" / "mnist"
partition_dir = data_dir / "partitions" / "Dirichlet"
data_dir.mkdir(parents=True, exist_ok=True)
partition_dir.mkdir(parents=True, exist_ok=True)

if ptype == "dirichlet":
    stem = f"dirichlet_alpha{ALPHA}_{NUM_CLIENTS}clients_seed{SEED}"
else:
    stem = f"noniid_{classes_per_client}_{NUM_CLIENTS}clients_seed{SEED}"

partition_path = partition_dir / f"{stem}.pt"
chart_path     = partition_dir / f"{stem}_dist.png"

print(f"Project root   : {project_root}")
print(f"Partition file : {partition_path}")

# ── Kiểm tra tồn tại ──────────────────────────────────────────────────────────
if partition_path.exists() and not OVERWRITE:
    print("\n⚠️  File partition đã tồn tại. Đặt OVERWRITE = True để tạo lại.")
else:
    # ── Load MNIST ────────────────────────────────────────────────────────────
    print("\nĐang tải MNIST …")
    train_ds     = datasets.MNIST(data_dir, train=True, download=True)
    train_labels = train_ds.targets.long()
    print(f"Tải xong: {len(train_labels):,} mẫu, 10 class.")

    # ── Tạo partition ─────────────────────────────────────────────────────────
    if ptype == "dirichlet":
        print(f"\nDirichlet: α={ALPHA}, K={NUM_CLIENTS}, seed={SEED}")
        client_indices, client_class_counts = generate_dirichlet_partition(
            train_labels, NUM_CLIENTS, ALPHA, SEED, MIN_SAMPLES
        )
        chart_title = f"Dirichlet (α={ALPHA}, K={NUM_CLIENTS}, seed={SEED})"
        extra_meta  = {"alpha": ALPHA, "classes_per_client": None}

    else:  # noniid_N
        print(f"\n{PARTITION_TYPE}: {classes_per_client} class/client, K={NUM_CLIENTS}, seed={SEED}")
        client_indices, client_class_counts = generate_noniid_partition(
            train_labels, NUM_CLIENTS, classes_per_client, SEED
        )
        chart_title = f"Non-IID {classes_per_client} class/client (K={NUM_CLIENTS}, seed={SEED})"
        extra_meta  = {"alpha": None, "classes_per_client": classes_per_client}

    # ── Lưu partition ─────────────────────────────────────────────────────────
    client_sizes = [len(x) for x in client_indices]
    torch.save(
        {
            "client_indices" : client_indices,
            "class_counts"   : client_class_counts,
            "client_sizes"   : client_sizes,
            "distribution"   : PARTITION_TYPE,
            "num_clients"    : NUM_CLIENTS,
            "seed"           : SEED,
            **extra_meta,
        },
        partition_path,
    )
    print(f"\n✅ Đã lưu: {partition_path}")

    # ── Thống kê ──────────────────────────────────────────────────────────────
    print(f"\n{'─'*42}")
    print(f"  Partition     : {PARTITION_TYPE}")
    print(f"  Num clients   : {NUM_CLIENTS}")
    print(f"  Total samples : {sum(client_sizes):,}")
    print(f"  Min / Max / Mean : {min(client_sizes)} / {max(client_sizes)} / {np.mean(client_sizes):.1f}")
    print(f"  Std           : {np.std(client_sizes):.1f}")
    print(f"{'─'*42}")

    # ── Vẽ biểu đồ ───────────────────────────────────────────────────────────
    if not NO_PLOT:
        plot_distribution(client_class_counts, chart_title, chart_path)

## 💾 Xuất CSV thống kê class

In [ ]:
partition   = torch.load(partition_path, map_location="cpu", weights_only=False)
counts_mat  = partition["class_counts"]     # (num_clients, num_classes)
num_classes = counts_mat.shape[1]
csv_path    = partition_path.with_name(partition_path.stem + "_class_counts.csv")

fieldnames = ["client_id", "total_samples"] + [f"class_{c}" for c in range(num_classes)]

with csv_path.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=fieldnames)
    writer.writeheader()
    for cid, row in enumerate(counts_mat.tolist()):
        writer.writerow({
            "client_id"     : cid,
            "total_samples" : sum(row),
            **{f"class_{c}": v for c, v in enumerate(row)},
        })

print(f"CSV đã lưu: {csv_path}")